In [12]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.decomposition import PCA

In [2]:
file_path = Path('../HC/data/km_data_2.xlsx')
sheets_dict = pd.read_excel(file_path, sheet_name=None, index_col=0)
print(sheets_dict.keys())

dict_keys(['data_kms'])


In [3]:
data_v1 = sheets_dict['data_kms']
clean_data_v1 = data_v1.dropna(axis=0, how='any')
clean_data_v1

,10y,10yMinus2y,VVIX,dVVIX,corr(SPX_10y)
Date,,,,,
2006-05-23,5.041,0.069,88.66,-0.033784,-0.328573
2006-05-24,5.040,0.106,88.63,-0.000338,-0.413760
2006-05-25,5.076,0.111,82.64,-0.067584,-0.419205
2006-05-26,5.054,0.105,72.50,-0.122701,-0.357171
2006-05-30,5.087,0.106,82.14,0.132966,-0.185569
...,...,...,...,...,...
2024-01-08,4.008,-0.153,81.26,-0.019428,-0.839615
2024-01-16,4.064,-0.168,89.59,0.102510,-0.690716
2024-01-17,4.100,-0.250,91.36,0.019757,-0.481139


In [4]:
model = Pipeline([
    ("normalization", StandardScaler()), 
    ("cluster", KMeans(n_clusters=3))
    ])

In [5]:
model.fit(clean_data_v1.values)
labels = model.predict(clean_data_v1.values)

In [6]:
labels_df = pd.DataFrame(labels, columns=['label'])
labels_df

,label
0,2
1,2
2,2
3,2
4,2
...,...
3939,2
3940,2
3941,2
3942,2


In [7]:
#### to plot cluster let's apply PCA factor scores

In [8]:
scaler = StandardScaler()
scaled_km_data = scaler.fit_transform(clean_data_v1.values)

In [9]:
pca = PCA()
pca.fit(scaled_km_data)

PCA()

In [10]:
eigenvectors_yields_sk = pca.components_
explained_variance = pca.explained_variance_ratio_
explained_variance

array([0.30152328, 0.25784914, 0.19650573, 0.16126058, 0.08286127])

In [11]:
km_pca_df = pd.DataFrame(pca.components_[:3], columns=clean_data_v1.columns, index=['PC1', 'PC2', 'PC3'])
km_pca_df

,10y,10yMinus2y,VVIX,dVVIX,corr(SPX_10y)
PC1,-0.635839,-0.127129,0.715436,0.233963,0.113841
PC2,-0.252000,0.695887,-0.137588,-0.192319,0.629539
PC3,0.217930,0.278137,-0.069384,0.931617,0.049223


In [13]:
fig_pcs_2 = go.Figure()

for pc in km_pca_df.index:
    fig_pcs_2.add_trace(
        go.Scatter(
            x=km_pca_df.columns,
            y=km_pca_df.loc[pc],
            mode='lines + markers',
            line={'dash':'dash'},
            name=str(pc),
        )
    )
    
    fig_pcs_2.update_layout(
        title='Eigenvector Loadings for selected eigenvectors',
        xaxis={'title': 'Variables'},
        yaxis={'title': 'Loadings'},
        template='plotly_white',
        width=1000,
        height=700,
    )

fig_pcs_2.show()

#### Factor Scores

In [17]:
y_pca_score = pca.transform(scaled_km_data)
y_pca_score

array([[-1.73597322, -1.78915407, -0.40859859,  0.58345954,  0.94358662],
       [-1.622652  , -1.96945134,  0.13542881,  0.46680144,  0.86191661],
       [-2.17999464, -1.70597088, -0.91902191,  0.389252  ,  0.87383356],
       ...,
       [-0.81933299, -2.15710989,  0.1479902 ,  0.23491956,  0.25886017],
       [-1.33423099, -1.64968694, -1.00581741,  0.40007635,  0.30566272],
       [-1.3997257 , -1.09129609, -0.88851901,  1.0575262 , -0.0255983 ]])

In [23]:
y_pca_df = pd.DataFrame(y_pca_score[:, :3], index=clean_data_v1.index, columns=['PC1', 'PC2', 'PC3'])
y_pca_df['label'] = labels_df['label'].values
y_pca_df

,PC1,PC2,PC3,label
Date,,,,
2006-05-23,-1.735973,-1.789154,-0.408599,2
2006-05-24,-1.622652,-1.969451,0.135429,2
2006-05-25,-2.179995,-1.705971,-0.919022,2
2006-05-26,-2.818264,-1.366825,-1.768972,2
2006-05-30,-1.345715,-2.117469,2.349504,2
...,...,...,...,...
2024-01-08,-1.446501,-2.251295,-0.466762,2
2024-01-16,-0.590853,-2.586305,1.491681,2
2024-01-17,-0.819333,-2.157110,0.147990,2


In [27]:
fig_fcs_2 = go.Figure()

for fcs in y_pca_df.columns:
    if fcs != 'label':
        if fcs == 'PC1':
            fig_fcs_2.add_trace(
                go.Scatter(
                    x=y_pca_df['PC2'],
                    y=y_pca_df[fcs],
                    mode='markers',
                    #text=[f"{p:.2f}%" for p in factor_scores_df[fcs]],
                    name=fcs,
                )
            )
        elif fcs == 'PC2':
            fig_fcs_2.add_trace(
                go.Scatter(
                    x=y_pca_df['PC1'],
                    y=y_pca_df[fcs],
                    mode='markers',
                    #text=[f"{p:.2f}%" for p in factor_scores_df[fcs]],
                    name=fcs,
                )
            )
        
    fig_fcs_2.update_layout(
        title='Factor Scores plot to show clusters',
        # xaxis={'title': 'Maturities', 'showgrid': False},
        # yaxis={'title': 'Change in Yields', 'showgrid': False},
        legend={'title': 'Factors Scores'},
        template='plotly_white',
        width=950,
        height=600,
    )

fig_fcs_2.show()